In [ ]:
! pip install mdutils
! pip install graphviz
! pip install titlecase

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for titlecase: filename=titlecase-2.4.1-py2.py3-none-any.whl size=10954 sha256=73fea563ba6aa3ea722d7e562676dc4633146b88ff61d3841297655bef27c40e
  Stored in directory: /root/.cache/pip/wheels/95/4d/41/317989157fb8bb4b7f1f00ab44a7594d578d68f2d1dd7ba0e7
Successfully built titlecase


In [ ]:
from titlecase import titlecase
import copy
import json

# Define a list of acronyms and special terms to preserve
def preserve_acronyms(word, **kwargs):
    """Callback function to preserve specific acronyms in their proper form."""
    acronym_map = {
        'genai': 'GenAI',
        'ai': 'AI',
        'xai': 'XAI',
        '—': '—'
    }
    return acronym_map.get(word.lower(), None)


def fix_code_names(schema):
    """Fix code names in the schema to follow the standard title case format."""
    # Create a deep copy to avoid modifying the original
    fixed_schema = copy.deepcopy(schema)

    # Iterate through each category
    for category_key, category_value in fixed_schema.items():
        if "codes" in category_value:
            # Create new codes dictionary with fixed names
            old_codes = category_value["codes"]
            new_codes = {}
            for code_name, code_description in old_codes.items():
                new_code_name = titlecase(code_name, callback=preserve_acronyms)
                new_codes[new_code_name] = code_description
            category_value["codes"] = new_codes

    return fixed_schema

# print(json.dumps(fix_code_names(futurework_schema), indent=4))

In [ ]:
import re
from collections import defaultdict
import pandas as pd
from typing import List, Dict, DefaultDict, Optional
from mdutils.mdutils import MdUtils


file_path = 'Tertiary study - data extraction.csv'
excerpt_column_name = 'Reported GenAI challenges/limitations (with code annotation)'
# excerpt_column_name = 'Suggestions about research gaps or future research directions  (with code annotation)'
# excerpt_column_name = 'Reported GenAI benefits with (with code annotation)'

challenges_schema = {
    "C1": {
        "name": "Societal, Ethical and Fairness Concerns",
        "description": "This category encompasses broad ethical and societal impacts. GenAI models amplify societal biases from training data, leading to discrimination. A major focus is the potential for misuse in generating harmful content like misinformation, propaganda, and deepfakes at scale. Concerns also include the erosion of human skills and autonomy, the risk of widening socio-economic disparities, and the significant environmental footprint from high energy consumption.",
        "description_short": "This category encompasses the broader ethical dilemmas and societal impacts of GenAI, addressing how it can perpetuate inequalities, harm individuals or communities, and raise moral responsibilities in deployment.",
        "codes": {
            "Biases and Discrimination": "The model's tendency to amplify societal biases (e.g., racial, gender, cultural) from its training data, leading to stereotypical or discriminatory outputs.",
            "Fairness Implementation Challenges": "Barriers to achieving fairness, including lack of consensus on standards across cultures and conflicts with efficiency or profitability goals.",
            "Potential for Misuse and Harmful Content": "The risk of GenAI being deliberately exploited to generate malicious, manipulative, or harmful content, including misinformation, deepfakes, hate speech, and phishing attacks. This raises issues of moral responsibility and the challenge of content moderation.",
            "Human Autonomy and Skill Devaluation": "Erosion of human skills, relationships, or dignity due to over-reliance, dehumanization, or automation (e.g., reduced critical thinking or social bonds).",
            "Societal and Economic Disparities": "The potential for GenAI to create or worsen societal and economic inequalities, including job displacement, widening the digital divide, and promoting market monopolization.",
            "Environmental Sustainability": "High energy consumption, carbon footprint or other resource demands of training/operating GenAI."
        }
    },
    "C2": {
        "name": "Reliability and Accuracy",
        "description": "This category focuses on technical shortcomings that undermine the dependability and precision of GenAI. Core challenges include hallucinations (plausible but fabricated information), outdated knowledge from fixed training cut-offs, and limited contextual awareness resulting in inappropriate outputs. Reliability is further compromised by performance drift and inconsistency over time. These risks are amplified by a pervasive lack of rigorous validation frameworks and domain-specific benchmarks.",
        "description_short": "This category focuses on the technical shortcomings that undermine the dependability and precision of GenAI outputs, highlighting risks in real-world applications where accuracy is critical.",
        "codes": {
            "Hallucinations and Factual Inaccuracy": "Generation of plausible but incorrect or misleading information",
            "Outdated or Limited Knowledge": "Fixed training cut‑off, sparse/low‑quality data, or inability to recall key information leading to outdated or incomplete responses.",
            "Limited Contextual Awareness": "Failure to grasp situational, cultural, or domain‑specific context, leading to outputs that are irrelevant, inappropriate, or superficial.",
            "Performance Inconsistency and Drift": "Unpredictable degradation or variability in model performance over time or across different scenarios, necessitating continuous monitoring.",
            "Lack of Evidence and Validation": "Scarcity of empirical studies, domain‑specific benchmarks, or validation."
        }
    },
    "C3": {
        "name": "Transparency and Explainability",
        "description": "This category addresses challenges in making GenAI systems understandable and auditable. The fundamental issue is the black-box nature of transformer architectures, which hinders interpretation, auditing, and trust. This opacity is compounded by the profound immaturity of current explainability (XAI) methods, which remain insufficient for revealing how models arrive at their outputs.",
        "description_short": "This category deals with the challenges in making GenAI systems understandable and auditable, emphasizing barriers to insight into their operations and decision-making.",
        "codes": {
            "Model Opacity (Black-Box Nature)": "The inherent lack of visibility into the internal workings and decision-making processes of the model, making it difficult to understand or audit.",
            "Explainability Shortcomings": "Immature or insufficient methods for explaining models (e.g., XAI gaps)."
        }
    },
    "C4": {
        "name": "Privacy, Security, Governance and Legal",
        "description": "This category covers the vulnerabilities and regulatory hurdles for GenAI. It details risks to data privacy through information leakage and system susceptibility to security breaches from malicious attacks like prompt injection. Underpinning these technical risks is a significant regulatory and legal gap, which in turn creates critical ambiguity around accountability and liability when harm occurs. The category also examines profound challenges to intellectual property regarding both training data and AI-generated content.",
        "description_short": "This category covers the vulnerabilities and regulatory hurdles in protecting data, ensuring system integrity, and establishing clear accountability for GenAI use.",
        "codes": {
            "Data Privacy Violations": "Unintended leakage or non‑consensual exposure of sensitive, personal, or proprietary information.",
            "Security Vulnerabilities": "Susceptibility to attacks such as prompt‑injection, data‑poisoning, or jailbreaks.",
            "Regulatory and Legal Gaps": "Absence of clear laws, standards or governance frameworks to ensure safe, ethical deployment of GenAI.",
            "Accountability and Liability Ambiguity": "Unclear assignment of responsibility or legal liability when harms arise from AI outputs or decisions.",
            "Copyright and Ownership Disputes": "Challenges related to ownership and rights, including the use of copyrighted material in training data and ambiguity over the legal status of AI-generated content."
        }
    }
}

futurework_schema = {
    "F1": {
        "name": "User perspective",
        "description": "This perspective encompasses several aspects, from designing effective means of human-AI interaction (e.g. multimodal interfaces) and recognizing user’s individual, through risks affecting individual users and GenAI’s psychological influence on humans to the skills necessary to use GenAI for particular purposes and the ways of educating people to use GenAI more effectively and safely.",
        "codes": {
            "User-GenAI interaction design": "Investigating how to design the effective user interfaces supporting humans in their tasks, with respect to various interaction modes (e.g. chatbots, voice assistants, multimodal interfaces) and expected behavior (e.g. recognizing irony or language nuances).",
            "The required skills and education": "Identifying the knowledge and skills people need to leverage GenAI and the ways to educate them.",
            "Personalization tailored to the individual user": "Investigating how to provide the individual user with the personalized content consistent with their needs, values or background.",
            "Psychological effects of GenAI use on individuals": "Investigating what psychological effects the continuous and/or long-term interaction with GenAI can have on individuals.",
            "Risks affecting individual users": "Exploring the risks GenAI can pose to individual users e.g. overreliance, misinformation."
        }
    },
    "F2": {
        "name": "Organizational perspective",
        "description": "This is the perspective of a company, enterprise or other organization using GenAI in its business processes. It includes identification of the factors important to the acceptance and adoption of GenAI by the organizations, its employees and other stakeholders. The key decision is the composition of human-AI teams including the assignments of tasks and responsibilities as well as dynamics of team’s operations. This requires decisions about which tasks should be automated and which left to be performed by humans. Moreover, adopting GenAI usually requires transforming the organization and changing its processes and structures. Finally, the possible benefits and threats introduced to an organization by GenAI are a viable topic of future research.",
        "codes": {
            "GenAI adoption and acceptance factors": "Identifying factors that affect the adoption and acceptance of GenAI solutions by organizations and their stakeholders.",
            "Formation of hybrid human-AI teams": "Investigating the organization, dynamics, and responsibility assignments of teams comprising humans and AI systems.",
            "Automation of routine tasks": "Investigating how some tasks, especially routine ones, can be redefined to benefit from automation delivered by GenAI.",
            "Organizational transformations to incorporate GenAI": "Investigating what transformations, including business processes and structures, are necessary to enable the organization to seamlessly exploit GenAI capabilities.",
            "Benefits provided by GenAI to organisations": "Understanding the benefits of GenAI for the organization, especially the competitive advantage it provides.",
            "Risks introduced by GenAI to organizations": "Understanding the risks organizations face when adopting and integrating GenAI."
        }
    },
    "F3": {
        "name": "Societal perspective",
        "description": "This perspective focuses on the consideration in what ways GenAI can and should impact societies (professions, social groups, business domains/sectors, nations, countries). A crucial aspect identified here is the specificity of different domains and business sectors which results in the need for investigating the specific context factors and the adjustments necessary for effective use of GenAI in a given domain/sector. The identified needs for GenAI regulations as well as the related issues of copyrights/propriety rights and legal challenges also form a promising area for research. The new, previously unknown use cases of GenAI that can bring new value as well as societal risks GenAI contributes to are directions worth investigating. Finally, the impact of GenAI on job market i.e. replacement of jobs by GenAI solutions but also the emergence of new jobs related to GenAI usage are included in this perspective.",
        "codes": {
            "Societal impact of GenAI": "Analyzing the impact of GenAI on social groups and societies and shaping it towards positive outcome.",
            "Application to specific domains and business sectors": "Identifying the context factors and the adjustments necessary for effective use of GenAI in specific domains and business sectors (e.g. healthcare, government, tourism, software development).",
            "Regulations and legal issues related to GenAI": "Investigating what regulations sanctioning GenAI are needed and new legal issues brought by this technology (e.g. copyrights and property rights).",
            "Risks introduced by GenAI to societies": "Understanding the risks affecting social groups and societies that can be introduced by GenAI.",
            "New use cases for GenAI applications": "Identifying new use cases that are likely to affect domains and business sectors and related social groups.",
            "Effect on job market": "Analyzing how the job market is affected by GenAI (e.g. jobs replaced by GenAI, new jobs created to utilize GenAI)."
        }
    },
    "F4": {
        "name": "Ethical perspective",
        "description": "The ethical issues related to GenAI are widely discussed as further research topics. Many sources indicate the need for researching the ways of ensuring that GenAI outcome is transparent with respect to algorithm, data, rationale and that its meaning is well-explained. Another widely recognized ethical issue is the possible bias and the effective ways of bias mitigation. Another issue is the requirement that the GenAI solution needs to be aware of user's specifics (e.g. national, cultural, ethnical) in order to provide the outcome suitable for such user instead of e.g. the outcome relevant only to the countries of the highest income. Also, GenAI solutions should be trained on data that mirrors diversity in the real world and can be effectively used by users of different backgrounds and abilities.",
        "codes": {
            "Ethical use of GenAI": "Identifying and defining ethical principles and guidelines related to GenAI and its applications.",
            "GenAI bias mitigation": "Identifying bias sources and propagation mechanisms and applying countermeasures to ensure bias mitigation.",
            "Transparency and explainability": "Assuring that GenAI solutions can be understandable (with respect to their construction, information they use, and how they reach the conclusion) and can explain their conclusions to users.",
            "Awareness of user's specifics": "Exploring the intended differentiation used to provide results more suitable to the user (e.g. different regions and cultures).",
            "Proper representation and inclusivity": "Assuring that GenAI solutions are trained on data that mirrors diversity in the real world and can be effectively used by users of different background and abilities."
        }
    },
    "F5": {
        "name": "Engineering perspective",
        "description": "This is the perspective of engineers (software developers, data scientists) responsible for creating GenAI solutions. It includes the core issue of GenAI model creation and training. Also, the need for metrics that capture key properties of GenAI is clearly recognized (such metrics may differ from the ones known from systems not dealing with AI or be entirely new). Evaluation and validation of existing GenAI systems is identified as a research gap – by evaluation/validation   we mean applying the GenAI system to real context (business sector, organization) and real data and observing results in the long term rather than running it on a test set and computing metrics like F1. There is a perceived lack of such evaluation, which seems necessary for GenAI’s adoption in real life processes, especially in domains that require reliable evidence (e.g. medicine/healthcare). More research on development and maintenance of systems with GenAI components resulting in definitions of processes, techniques and good practices is expected, with the emphasis on design principles for such systems.",
        "codes": {
            "Definition of GenAI metrics": "Defining the new metrics that would make it possible to evaluate GenAI solutions with respect to their key aspects e.g. ethical principles, quality attributes.",
            "Empirical evaluation of GenAI in the field": "Designing and conducting evaluation studies in real-world settings and context (including real data, users, use of GenAI output in business processes).",
            "Design principles for GenAI solutions": "Establishing design principles and recommended practices for effective development of GenAI solutions.",
            "GenAI model training": "Exploring the ways of designing, training and GenAI models, including enabling them to work with evolving datasets.",
            "GenAI system development and maintenance": "Identifying the processes and engineering practices suitable for development and maintenance of GenAI systems."
        }
    },
    "F6": {
        "name": "Quality requirements perspective",
        "description": "This perspective encompasses several quality properties and related categories of quality requirements relevant for GenAI solutions and not listed among the previous perspectives. One of them is the trustworthiness of GenAI to its users i.e. they are willing to use it and consider its responses to be reliable. Other issues include data privacy and protection of users from security threats and various frauds and misuses of GenAI. The well-known properties like efficiency/performance and scalability are also considered important and requiring further research. Finally, the accountability of GenAI providers and the channels enabling users’ complaints about inappropriate GenAI output are among topics that require more investigation.",
        "codes": {
            "Data privacy protection": "Establishing the effective ways of protecting personal or sensitive data.",
            "Security and protection": "Identifying the means to protect users against not only security attacks but also frauds and malicious uses of GenAI.",
            "Demonstrating trustworthiness": "Identifying how to achieve and demonstrate GenAI's trustworthiness and reliability to its users.",
            "Efficiency and scalability": "Assuring that GenAI solutions respond in reasonable time and can be scalable with respect to e.g. number of users or volumes of data.",
            "Accountability and contestability": "Assuring that GenAI providers are accountable for the output generated by their solutions and the users have contestability mechanisms to complain about the output they received."
        }
    }
}

benefits_schema = {
     "B1": {
         "name": "Healthcare and Life Sciences",
         "description": "Benefits associated with healthcare, covering both patient care and medical practice — such as improved information flow, enhanced decision-making, and the use of technologies that help patients manage and improve their health.",
         "codes": {
             "Diagnostics, decision-making, and patient care": "Supports accurate diagnostics and  decision-making to enhance the quality and efficiency of patient care.",
             "Drug discovery acceleration and precision medicine": "Harnessing innovation to accelerate drug discovery and provide more precise, personalized treatments that improve patient outcomes.",
             "Clinical documentation and workflow": "Facilitates the development of clinical documentation and improves operational efficiency by streamlining workflows and tasks within individual units.",
             "Strengthening mental health": "Enhances emotional support and fosters psychological well-being to strengthen overall mental health and quality of life.",
             "Access to healthcare knowledge and simplified communication": "Information about health, diseases and treatment is more accessible to everyone, simplified communication of complex medical/statistical information."
         }
     },
	   "B2": {
         "name": "Education and Learning",
         "description": "Gen AI transforms education by accelerating learning, delivering personalized learning experiences, and facilitating easier access to knowledge.",
         "codes": {
             "Personalized learning": "Enables personalized learning experiences and automates the creation of educational content.",
             "Digital learning resources, reduced educator administrative tasks": "Creation of comprehensive digital learning resources, reducing and streamlining educators' work related to documentation, reporting and administrative formalities.",
             "Assistance in language translation and accessibility": "Assistance in translating texts and ensuring that content or services are easily accessible to all users, including those with different needs, e.g. people with disabilities."
         }
     },
	   "B3": {
        "name": "Research, Innovation, and Design",
         "description": "Gen AI streamlines research and development by supporting information analysis and the creation of innovative ideas, which accelerates the development of knowledge and innovation.",
         "codes": {
             "Supports design knowledge and research across disciplines": "Extraction and synthesis of information from various fields to create comprehensive design solutions, supporting design science research and design thinking with Gen  AI.",
             "Tool-supported idea generation": "The use of specialised software that supports creativity and helps to develop new, innovative solutions faster and more effectively."
         }
     },
	   "B4": {
         "name": "Software Engineering and Technical Productivity",
         "description": "Accelerates development, enhances code quality, and improves system performance by automating application development, testing, and optimization.",
         "codes": {
             "Analysis, coding, testing and translations (automatically)": "Supports efficient development by generating code, producing test cases, and enabling translation between different programming languages.",
             "Efficiency in software development": "Gen AI and LLMs streamline software development, enhancing productivity, efficiency, decision-making and email management."
         }
     },
	   "B5": {
         "name": "Communication, Accessibility, Services, and Social Impact",
         "description": "Improves communication through automatic translations and summaries, and improves digital administration services by offering a better user experience.",
         "codes": {
             "Communication and accessibility": "Facilitates bridging communication gaps, delivering tailored services to diverse audiences and needs, and fostering positive societal impact by enhancing engagement and understanding.",
             "Enhanced user interaction and experience": "Communication between the user and the system becomes more natural, fast and effective, which makes the use of services or products more intuitive and tailored to individual needs.",
			 "Digital government services": "Enhanced digital government services through translation and accessibility."
         }
     },
	   "B6": {
         "name": "Data Management, Creative Content, Privacy & Ethics",
         "description": "Supports the collection, organisation and structuring of information and accelerates the creation of new content, which streamlines work and increases creativity. Data management, privacy, and ethics ensure the safe, lawful, and responsible use of AI, protecting users and their data.",
         "codes": {
             "Summaries and notes": "Automatic summarization and note creation.",
             "Content creation": "Creation of new text, image, audio, and video content.",
			 "Synthetic data, reducing bias, increasing responsibility": "Synthetic data generation for privacy preservation, reduction of harmful biases and enhanced corporate responsibility.",
			 "Task automation and service scaling": "The automation of routine tasks enhances productivity, supports innovation, and enables scalable and efficient business processes."
          }
     }
}



themes = {
    "B": {
        "name": "Benefits",
        "excerpt_column_name": "Reported GenAI benefits with (with code annotation)",
        "contributors": "Jacinto Estima and Beata Zielosko",
        "category_prefix": "B",
        "schema": benefits_schema,
        "file_name": "benefits.md"
    },
    "C": {
        "name": "Challenges and Limitations",
        "excerpt_column_name": "Reported GenAI challenges/limitations (with code annotation)",
        "contributors": "Yen Ying Ng and Adam Przybylek",
        "category_prefix": "C",
        "schema": challenges_schema,
        "file_name": "challenges.md"
    },
    "F": {
        "name": "Research Gaps and Future Research Directions",
        "excerpt_column_name": "Suggestions about research gaps or future research directions  (with code annotation)",
        "contributors": "Aleksander Jarzebowicz and Jakub Swacha",
        "category_prefix": "F",
        "schema": futurework_schema,
        "file_name": "future_research.md"
    }
}

theme = themes["C"]


In [ ]:
from graphviz import Digraph

def smart_wrap(text, max_len=35):
    """Smart text wrapping that creates balanced lines"""
    if len(text) <= max_len:
        return text

    # Try to split at comma first
    if ',' in text:
        parts = text.split(',', 1)
        if len(parts[0]) > 15 and len(parts[0]) < 40:
            return parts[0] + ',\n' + parts[1].strip()

    # Try to split at ' and '
    if ' and ' in text:
        parts = text.split(' and ', 1)
        if abs(len(parts[0]) - len(parts[1])) < 20:
            return parts[0] + '\nand ' + parts[1]

    # Split at roughly middle, but at word boundary
    words = text.split()
    if len(words) <= 2:
        return text

    # Find the split point that creates most balanced lines
    total_len = len(text)
    target = total_len / 2

    current_len = 0
    best_split = len(words) // 2
    best_diff = float('inf')

    for i in range(1, len(words)):
        current_len = len(' '.join(words[:i]))
        diff = abs(current_len - target)
        if diff < best_diff:
            best_diff = diff
            best_split = i

    return ' '.join(words[:best_split]) + '\n' + ' '.join(words[best_split:])


def create_theme_tree(schema, theme_name, output_format='eps'):
    """
    Create a tree visualization for a given theme schema.

    Parameters:
    -----------
    schema : dict
        The schema dictionary containing categories and codes
    theme_name : str
        Name of the theme (e.g., 'Challenges', 'Future Research', 'Benefits')
    output_format : str
        Output format: 'png', 'eps', or 'pdf' (default: 'eps')

    Returns:
    --------
    Digraph object
    """
    dot = Digraph(comment=theme_name, format=output_format)
    dot.attr(rankdir='LR', splines='polyline', nodesep='0.2', ranksep='0.8')
    dot.attr('node', shape='box', style='rounded,filled', fontname='Arial', margin='0.1,0.05')
    dot.attr('edge', dir='none')  # No arrows
    dot.attr(dpi='300')

    # **Minimal margins for PDF format to prevent clipping**
    if output_format == 'pdf':
        dot.attr(margin='0.1', pad='0.1')
    else:
        dot.attr(size='10,14!')

    # Root node - grayscale
    root_label = theme_name.replace(' ', '\n') if len(theme_name.split()) == 2 else theme_name
    dot.node('root', root_label, fillcolor='#4a4a4a', fontcolor='white',
             fontsize='12', style='rounded,filled,bold', width='1.8', height='0.4')

    # Add categories and codes
    for cat_id, cat_data in schema.items():
        cat_node = f'cat_{cat_id}'
        # NO category ID - just the name with smart wrapping
        cat_label = smart_wrap(cat_data['name'], max_len=35)

        dot.node(cat_node, cat_label, fillcolor='#d3d3d3', fontcolor='black',
                fontsize='9', width='2.2', height='0.6')
        dot.edge('root', cat_node, penwidth='1.5', color='#6a6a6a')

        # Add codes for this category
        for i, code_name in enumerate(cat_data['codes'].keys()):
            code_node = f'code_{cat_id}_{i}'
            wrapped_name = smart_wrap(code_name, max_len=40)
            dot.node(code_node, wrapped_name, fillcolor='white', fontcolor='black',
                    fontsize='8', width='2.5', height='0.4')
            dot.edge(cat_node, code_node, penwidth='1', color='#999999')

    return dot


def create_future_research_tree_split(schema, theme_name, output_format='eps'):
    """
    Create a bidirectional layout for Future Research with 3 categories on each side.
    Root in middle, 3 categories LEFT, 3 categories RIGHT.

    Parameters:
    -----------
    schema : dict
        The schema dictionary containing categories and codes
    theme_name : str
        Name of the theme
    output_format : str
        Output format: 'png', 'eps', or 'pdf' (default: 'eps')

    Returns:
    --------
    Digraph object
    """
    dot = Digraph(comment=theme_name, format=output_format)
    dot.attr(rankdir='LR', splines='polyline', nodesep='0.2', ranksep='0.8')
    dot.attr('node', shape='box', style='rounded,filled', fontname='Arial', margin='0.1,0.05')
    dot.attr('edge', dir='none')
    dot.attr(dpi='300')

    # **Minimal margins for PDF format to prevent clipping**
    if output_format == 'pdf':
        dot.attr(margin='0.1', pad='0.1')
    else:
        dot.attr(size='14,10!')

    # Root node - center
    root_label = theme_name.replace(' ', '\n') if len(theme_name.split()) == 2 else theme_name
    dot.node('root', root_label, fillcolor='#4a4a4a', fontcolor='white',
             fontsize='12', style='rounded,filled,bold', width='1.8', height='0.4')

    categories = list(schema.items())

    # LEFT SIDE: Last 3 categories (F4, F5, F6) - note reversed edge direction
    for cat_id, cat_data in categories[3:]:  # F4, F5, F6
        cat_node = f'cat_{cat_id}'
        cat_label = smart_wrap(cat_data['name'], max_len=35)

        dot.node(cat_node, cat_label, fillcolor='#d3d3d3', fontcolor='black',
                fontsize='9', width='2.2', height='0.6')
        # REVERSED: category -> root (makes category appear on LEFT)
        dot.edge(cat_node, 'root', penwidth='1.5', color='#6a6a6a')

        # Add codes extending further LEFT
        for i, code_name in enumerate(cat_data['codes'].keys()):
            code_node = f'code_{cat_id}_{i}'
            wrapped_name = smart_wrap(code_name, max_len=40)
            dot.node(code_node, wrapped_name, fillcolor='white', fontcolor='black',
                    fontsize='8', width='2.5', height='0.4')
            # REVERSED: code -> category
            dot.edge(code_node, cat_node, penwidth='1', color='#999999')

    # RIGHT SIDE: First 3 categories (F1, F2, F3) - normal direction
    for cat_id, cat_data in categories[:3]:  # F1, F2, F3
        cat_node = f'cat_{cat_id}'
        cat_label = smart_wrap(cat_data['name'], max_len=35)

        dot.node(cat_node, cat_label, fillcolor='#d3d3d3', fontcolor='black',
                fontsize='9', width='2.2', height='0.6')
        # NORMAL: root -> category (makes category appear on RIGHT)
        dot.edge('root', cat_node, penwidth='1.5', color='#6a6a6a')

        # Add codes extending further RIGHT
        for i, code_name in enumerate(cat_data['codes'].keys()):
            code_node = f'code_{cat_id}_{i}'
            wrapped_name = smart_wrap(code_name, max_len=40)
            dot.node(code_node, wrapped_name, fillcolor='white', fontcolor='black',
                    fontsize='8', width='2.5', height='0.4')
            # NORMAL: category -> code
            dot.edge(cat_node, code_node, penwidth='1', color='#999999')

    return dot


def generate_all_themes(output_format='eps', view=False):
    """
    Generate visualizations for all three themes.

    Parameters:
    -----------
    output_format : str
        Output format: 'png', 'eps', or 'pdf' (default: 'eps')
    view : bool
        Whether to open the generated files (default: False)

    Returns:
    --------
    None
    """
    # Generate Challenges (standard layout)
    dot_challenges = create_theme_tree(challenges_schema, 'Challenges', output_format)
    dot_challenges.render('genai_challenges', view=view, cleanup=True)
    print(f"✓ Created: genai_challenges.{output_format}")

    # Generate Future Research (split left-right layout)
    dot_future = create_future_research_tree_split(futurework_schema, 'Future Research', output_format)
    dot_future.render('genai_future_research', view=view, cleanup=True)
    print(f"✓ Created: genai_future_research.{output_format}")

    # Generate Benefits (standard layout)
    dot_benefits = create_theme_tree(benefits_schema, 'Benefits', output_format)
    dot_benefits.render('genai_benefits', view=view, cleanup=True)
    print(f"✓ Created: genai_benefits.{output_format}")

    print(f"\n✅ All three {output_format.upper()} files created!")


# ==================== USAGE ====================

generate_all_themes(output_format='eps')
# generate_all_themes(output_format='pdf')
# generate_all_themes(output_format='png')

✓ Created: genai_challenges.eps
✓ Created: genai_future_research.eps
✓ Created: genai_benefits.eps

✅ All three EPS files created!


In [ ]:
def load_excerpts_from_csv(filepath: str, excerpt_column: str) -> Optional[pd.DataFrame]:
    try:
        df = pd.read_csv(filepath, usecols=["Assigned to:","Paper ID:","Paper:","Group",excerpt_column], dtype=str)
        df.fillna("", inplace=True)
        df.rename(columns={"Assigned to:":"Extractor","Paper ID:": "ID", "Paper:": "Paper", excerpt_column: "Excerpt"}, inplace=True)
        # [SR]   - matches a single character: either 'S' or 'R'.
        # \d+    - matches one or more digits.
        pattern = r"^[SR]\d+$"
        mask = df["ID"].str.match(pattern)
        df = df[mask].reset_index(drop=True)
        df["SLR"] = df["ID"].str.startswith("S")
        return df
    except FileNotFoundError:
        print(f"Error: The file '{filepath}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None


def process_excerpts(excerpts_DF: pd.DataFrame, theme: Dict) -> Dict[str, List[str]]:
    """
    Processes a DataFrame to find all occurrences of codes and maps them to the Paper IDs in which they appear.

    Args:
        excerpts_DF (pd.DataFrame): DataFrame containing 'ID' and 'Excerpt' columns.
        theme (Dict): The theme defining valid codes and other thinks.


    Returns:
        Dict[str, List[str]]: A dictionary where keys are the codes and values
                              are lists of unique Paper IDs where the code was found.
    """
    # Use a defaultdict of sets to automatically handle unique Paper IDs
    code_to_ids_map = defaultdict(set)

    # Flatten the schema for efficient validation
    all_valid_codes = {code for category_data in theme["schema"].values() for code in category_data['codes'].keys()}

    # Build the regex pattern dynamically
    category_prefix = theme["category_prefix"]
    regex_pattern = r'\{(.*?)\s*\['+category_prefix+r'\d\]\}'

    # Iterate over each row of the DataFrame using .itertuples() for efficiency
    for row in excerpts_DF.itertuples(index=False):
        paper_id = row.ID
        excerpt_text = row.Excerpt

        matches = re.findall(regex_pattern, excerpt_text.strip())
        unique_codes_in_excerpt = set(matches)

        for code in unique_codes_in_excerpt:
            if code in all_valid_codes:
                # Add the current paper's ID to the set for this code
                code_to_ids_map[code].add(paper_id)
            else:
                print(f"Warning (Paper ID: {paper_id}): Unknown code found and ignored: '{code}'")

    # Convert the sets of IDs to sorted lists for clean, consistent output
    final_code_map = {code: sorted(list(ids)) for code, ids in code_to_ids_map.items()}

    return final_code_map

excerpts_DF = load_excerpts_from_csv(file_path, theme["excerpt_column_name"])
# print(excerpts_DF)

if excerpts_DF is not None:
    # Get the dictionary mapping codes to lists of Paper IDs
    code_map = process_excerpts(excerpts_DF, theme)

    print("\n--- Code Occurrences by Paper ID ---")

    # Iterate through the main schema dictionary
    for category_id, category_data in challenges_schema.items():
        # Extract the full category name from the nested structure
        category_full_name = category_data.get("name", category_id)
        print(f"\n{category_full_name}:")

        # Iterate through the 'codes' dictionary for this category
        for code in category_data['codes'].keys():
            # Get the list of IDs for the current code, or an empty list if not found
            paper_ids = code_map.get(code, [])
            # The count is simply the length of this list
            count = len(paper_ids)

            # Format the list of IDs for printing
            ids_str = ', '.join(paper_ids) if paper_ids else "None"

            print(f"  - {code} [{count}]: {ids_str}")


--- Code Occurrences by Paper ID ---

Societal, Ethical and Fairness Concerns:
  - Biases and Discrimination [13]: R01, R02, R04, R07, R08, R09, R10, S01, S08, S09, S11, S14, S16
  - Fairness Implementation Challenges [3]: R01, R08, S09
  - Potential for Misuse and Harmful Content [14]: R02, R03, R06, R07, R08, R10, S01, S02, S06, S08, S09, S14, S15, S17
  - Human Autonomy and Skill Devaluation [9]: R08, R09, S01, S08, S09, S11, S12, S14, S17
  - Societal and Economic Disparities [6]: R04, R07, R08, S01, S09, S11
  - Environmental Sustainability [8]: R01, R02, R04, R10, S07, S09, S10, S15

Reliability and Accuracy:
  - Hallucinations and Factual Inaccuracy [18]: R02, R03, R04, R05, R07, R08, R10, S01, S02, S03, S05, S07, S09, S10, S11, S14, S16, S17
  - Outdated or Limited Knowledge [6]: R04, R06, S07, S10, S11, S14
  - Limited Contextual Awareness [8]: R04, R05, R06, S01, S05, S07, S09, S11
  - Performance Inconsistency and Drift [5]: R04, S05, S07, S09, S11
  - Lack of Evidence and 

In [ ]:
def get_groups_for_papers(paper_ids: list[str], excerpts_DF: pd.DataFrame) -> pd.Series:
    lookup = excerpts_DF.set_index('ID')['Group']
    return lookup.reindex(paper_ids)

def modify_paperID_aesthetic(paper_id: str, paper_group: str) -> str:
    """Return a badge for the paper_id with a color based on its group."""
    group_colors = {
        "Education": "salmon",
        "Healthcare": "gold",
        "MIS": "aqua",
        "IS": "azure",
    }
    color = group_colors.get(paper_group)
    if color:
        return f"![{paper_id}](https://img.shields.io/badge/{paper_id}-{color}?style=plastic)"
    else:
        return paper_id

def generate_markdown_report(code_map: Dict[str, List[str]], excerpts_DF: pd.DataFrame, theme: Dict):
    """
    Generates a fully cross-referenced Markdown report.
    - Part 1 (Schema) has anchors for each code and is a continuous list.
    - Part 2 (Bibliography) has links from excerpts back to the schema,
      formatted as {*linked code*}.
    """
    mdFile = MdUtils(file_name=theme["file_name"])

    # --- Helper function for creating clean anchor links ---
    def create_anchor_link(text: str) -> str:
        # Converts "Some Code Name" to "some-code-name" for a URL-friendly anchor
        return re.sub(r'\s+', '-', text.lower().strip())

    # --- Part 1: Schema with Linked Paper IDs AND Anchors for Each Code ---
    mdFile.new_header(level=1, title=f"Theme: {theme['name']} [^1]")

    # --- Legend block ---
    legend_items = [
        f"**Legend:**",
        modify_paperID_aesthetic("IS", "IS") + " ",
        modify_paperID_aesthetic("MIS", "MIS") + " ",
        modify_paperID_aesthetic("Healthcare", "Healthcare") + " ",
        modify_paperID_aesthetic("Education", "Education") + " "
    ]
    # Put the legend into one paragraph, separated by spaces
    mdFile.new_paragraph("   ".join(legend_items))
    mdFile.new_line()



    for category_id, category_data in theme["schema"].items():
        category_name = category_data['name']
        category_description = category_data.get('description', '') # Use .get for safety

        mdFile.new_header(level=2, title=f"{category_id}: {category_name}")
        mdFile.new_paragraph(f"*{category_description}*")

        # Create an empty list to hold all list items for this category
        list_items = []
        for code, code_description in category_data['codes'].items():
            # Create an invisible HTML anchor for this specific code
            code_anchor = create_anchor_link(code)
            invisible_code_anchor = f'<a id="{code_anchor}"></a>'

            paper_ids = code_map.get(code, [])
            count = len(paper_ids)
            slr_count = sum(1 for pid in paper_ids if pid.startswith("S"))
            roadmap_count = sum(1 for pid in paper_ids if pid.startswith("R"))

            linked_ids_str = "None"
            if paper_ids:
                # Link to the paper's bibliography entry
                group_series = get_groups_for_papers(paper_ids, excerpts_DF)
                linked_ids = [mdFile.new_inline_link(link=f"#{paper_id}", text=modify_paperID_aesthetic(paper_id, group_series[paper_id])) for paper_id in paper_ids]
                # linked_ids = [mdFile.new_inline_link(link=f"#{paper_id}", text=paper_id) for paper_id in paper_ids]
                linked_ids_str = ", ".join(linked_ids)


            # Append the fully formatted string to our list of items.
            # The invisible anchor is placed at the start of the string.
            list_items.append(
                f"{invisible_code_anchor}**{code}** [{count} | roadmap: {roadmap_count} | SLR: {slr_count}]: {linked_ids_str}  \n"
                f"  *{code_description}*"
            )

        # Now, create the list once with all the items to ensure it's continuous
        mdFile.new_list(list_items, marked_with='-')

    mdFile.new_line()

    # --- Part 2: Bibliography with Excerpts (with links back to the schema) ---
    mdFile.new_header(level=1, title='Included Papers and Excerpts')

    # --- The regex replacement function ---
    def replace_annotation_with_link(match):
        # The first group (.*?) captures the code name
        # match.group(0): {Hallucinations and Inaccuracy [C2]}
        # match.group(1): Hallucinations and Inaccuracy
        # code_name = match.group(1).strip()
        #####code_name = match.group(0)
        # Create the anchor link that matches the one in Part 1
        anchor_link = create_anchor_link(match.group(1).strip())
        # Create the formatted Markdown link and wrap it in curly braces
        markdown_link = mdFile.new_inline_link(link=f"#{anchor_link}", text=f"*{match.group(0)}*")
        return f"{markdown_link}"
        # return f"{{{markdown_link}}}"

    sorted_df = excerpts_DF.sort_values(by=['ID'], ascending=[True]).reset_index(drop=True)
    # sorted_df = excerpts_DF.sort_values(by=['SLR','Group', 'ID'], ascending=[True, False, True]).reset_index(drop=True)

    for index, row in sorted_df.iterrows():
        paper_id = row['ID']
        group = row['Group']
        paper_reference = row['Paper']
        excerpt_text = row['Excerpt']
        extractor = row['Extractor']

        # Create the invisible anchor for this paper
        invisible_paper_anchor = f'<a id="{paper_id}"></a>'
        mdFile.write(invisible_paper_anchor)

        # Add the bibliography entry as a single list item
        # mdFile.new_list([f"**[{paper_id}]** {paper_reference}"])
        mdFile.new_list([f"{modify_paperID_aesthetic(paper_id, group)} **<mark>{paper_reference}</mark>** (Data extractor: *{extractor}*)"])


        if pd.notna(excerpt_text) and excerpt_text.strip():
            # Find and replace all annotations in the excerpt
            processed_excerpt = re.sub(
                r'\{(.*?)\s*\[[A-Z]\d\]\}',
                replace_annotation_with_link,
                excerpt_text
            )

            # Create the multi-line blockquote
            lines = processed_excerpt.strip().splitlines()
            quoted_lines = [f"> {line}  " for line in lines]
            blockquote_text = "\n".join(quoted_lines)
            mdFile.new_paragraph(blockquote_text)

        mdFile.new_paragraph("---")
        mdFile.new_paragraph()

    mdFile.new_paragraph(f"[^1]: Codebook Authors & Data Annotators: *{theme['contributors']}*")
    # --- Create the Markdown File ---
    mdFile.create_md_file()
    print(f"\nSuccessfully generated Markdown report: {theme['file_name']}")

In [ ]:
import re
def replace_annotation_with_link(match):
        print(match.group(0))
        code_name = match.group(1).strip()
        print(code_name)
        return code_name

excerpt_text = "its current limitations, particularly in clinical accuracy and the risk of generating misinformation {Hallucinations and Inaccuracy [C2]}, necessitate cautious integration into practice, with continuous oversight from healthcare professionals"
processed_excerpt = re.sub(
                r'\{(.*?)\s*\[[A-Z]\d\]\}',
                replace_annotation_with_link,
                excerpt_text
            )
print(processed_excerpt)

{Hallucinations and Inaccuracy [C2]}
Hallucinations and Inaccuracy
its current limitations, particularly in clinical accuracy and the risk of generating misinformation Hallucinations and Inaccuracy, necessitate cautious integration into practice, with continuous oversight from healthcare professionals


In [ ]:
for theme in themes.values():
    excerpts_DF = load_excerpts_from_csv(file_path, theme["excerpt_column_name"])

    if excerpts_DF is not None:
        code_map = process_excerpts(excerpts_DF, theme)

        # The generate_markdown_report function no longer needs the separate 'categories' dictionary.
        generate_markdown_report(
            code_map=code_map,
            excerpts_DF=excerpts_DF,
            theme=theme
        )


Successfully generated Markdown report: benefits.md

Successfully generated Markdown report: challenges.md

Successfully generated Markdown report: future_research.md
